# Actividad 3 · Distribuciones iniciales

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 2 pomodoros (40 minutos)

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

En esta actividad vas a describir distribuciones mediante consultas SQL, medidas de resumen y gráficos simples. Trabajaremos con nacimientos vivos (`outcome = 1`) y utilizaremos kilogramos para expresar el peso al nacer.


## Preparación automática

Ejecutá la celda siguiente sin modificarla. Preparará el dataset y la conexión con DuckDB; no forma parte de los contenidos evaluados.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/2002FemPreg.dat.gz"
DATA_PATH = Path("2002FemPreg.dat.gz")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

colspecs = [
    (0, 12), (12, 14), (56, 58), (58, 60), (274, 276),
    (276, 277), (277, 279), (283, 287), (422, 440),
]
names = [
    "caseid", "pregordr", "birthwgt_lb", "birthwgt_oz", "prglngth",
    "outcome", "birthord", "agepreg", "finalwgt",
]

embarazos = pd.read_fwf(
    DATA_PATH, names=names, colspecs=colspecs, compression="gzip"
)
embarazos["birthwgt_lb"] = embarazos["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)
embarazos["birthwgt_oz"] = embarazos["birthwgt_oz"].replace([97, 98, 99], np.nan)
embarazos["totalwgt_lb"] = embarazos["birthwgt_lb"] + embarazos["birthwgt_oz"] / 16
embarazos["totalwgt_kg"] = (embarazos["totalwgt_lb"] * 0.45359237).round(3)
embarazos["agepreg"] = embarazos["agepreg"] / 100
embarazos = embarazos[[
    "caseid", "pregordr", "prglngth", "outcome", "birthord",
    "birthwgt_lb", "birthwgt_oz", "totalwgt_lb", "totalwgt_kg",
    "agepreg", "finalwgt",
]]

con = duckdb.connect(":memory:")
con.register("embarazos_df", embarazos)
con.execute("CREATE OR REPLACE TABLE embarazos AS SELECT * FROM embarazos_df")
print(f"Entorno preparado: {len(embarazos):,} registros disponibles en la tabla 'embarazos'.")


## Cantidad de nacimientos vivos


In [ ]:
con.sql("""
SELECT COUNT(*) AS nacimientos_vivos
FROM embarazos
WHERE outcome = 1
""")


## Resumen del peso al nacer

Calculemos cantidad de casos, media, desvío estándar, mínimo y máximo del peso en kilogramos.


In [ ]:
con.sql("""
SELECT
    COUNT(totalwgt_kg) AS n,
    ROUND(AVG(totalwgt_kg), 2) AS promedio_kg,
    ROUND(STDDEV(totalwgt_kg), 2) AS desvio_estandar_kg,
    ROUND(MIN(totalwgt_kg), 2) AS minimo_kg,
    ROUND(MAX(totalwgt_kg), 2) AS maximo_kg
FROM embarazos
WHERE outcome = 1
""")


Estas medidas sintetizan la distribución, pero deben interpretarse junto con la cantidad de casos válidos y la forma observada en los gráficos.

## Distribución de la duración del embarazo


In [ ]:
duracion = con.sql("""
SELECT prglngth AS semanas, COUNT(*) AS cantidad
FROM embarazos
WHERE outcome = 1
GROUP BY prglngth
ORDER BY prglngth
""").df()
duracion


Con Python, representamos gráficamente el resultado obtenido mediante SQL.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.bar(duracion["semanas"], duracion["cantidad"])
plt.xlabel("Duración del embarazo (semanas)")
plt.ylabel("Cantidad")
plt.title("Distribución de la duración del embarazo")
plt.xlim(20, 50)
plt.show()


La mayor concentración se encuentra alrededor de las semanas habituales de gestación. Los valores alejados requieren atención, pero su sola presencia no permite decidir que sean errores.

## Distribución de la edad de la madre


In [ ]:
edades = con.sql("""
SELECT ROUND(agepreg, 0) AS edad, COUNT(*) AS cantidad
FROM embarazos
WHERE outcome = 1 AND agepreg IS NOT NULL
GROUP BY ROUND(agepreg, 0)
ORDER BY edad
""").df()
edades


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(edades["edad"], edades["cantidad"])
plt.xlabel("Edad de la madre (años)")
plt.ylabel("Cantidad")
plt.title("Distribución de la edad de la madre al finalizar el embarazo")
plt.show()


## Ejercicio

Construí un histograma del peso al nacer (`totalwgt_kg`) para los nacimientos vivos. Incluí un título y rotulá el eje horizontal como **Peso al nacer (kg)**. Luego escribí dos o tres oraciones que describan únicamente lo que muestra la distribución, sin proponer todavía explicaciones causales.


In [ ]:
# Escribí acá el código para construir el histograma


## Cierre

En esta semana describimos variables y reconocimos patrones iniciales. No realizamos todavía comparaciones formales entre grupos ni formulamos conclusiones definitivas: esas decisiones requieren nuevas preguntas y análisis posteriores.

**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1; VanderPlas, J. (2016). *Python Data Science Handbook*, capítulo 4.


In [ ]:
con.close()
